# 6 — Head to head: neural copula vs vine copula

Vine copulas are the established, well-engineered answer to multivariate dependence, and
[`pyvinecopulib`](https://github.com/vinecopulib/pyvinecopulib) is their reference
implementation. Any claim that a neural copula is worth using has to be made against them.

This notebook makes that comparison honestly, which means showing the cases where the flow
**loses** as prominently as the ones where it wins.

## How the two models differ

| | Vine copula | Neural copula |
|---|---|---|
| Structure | $d(d-1)/2$ bivariate pair copulas in a tree | One flow over all $d$ dimensions |
| Dependence shape | Chosen from a family catalogue | Learned |
| Parameters | 1–2 per pair | Tens of thousands |
| Interpretable | Yes — family and τ per pair | No |
| Fit cost | Fast | 30–100x slower |
| Fails when | No family in the set fits | Data is small relative to capacity |

A vine is a *structured, low-parameter* model. That is a strength when the structure is right
and a hard ceiling when it is not.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from neurocopula import NeuroCopula, VineCopula, datasets, metrics, plotting as ncplot, theme
from neurocopula.benchmark import _split

pd.set_option("display.width", 150, "display.precision", 4)

# Apply the library's visual theme, so hand-rolled figures in this notebook
# match the ones the library produces. rcParams are read when an Axes is
# created, so this must run before any plotting.
theme.apply_theme()

## The same API for both

`VineCopula` deliberately mirrors `NeuroCopula`, so the two can be compared without the
comparison becoming a comparison of call conventions.

In [ ]:
df = datasets.make_clayton(n=4000, theta=2.0, seed=0)
train, test = _split(df, 0.25, seed=0)

models = {
    "NeuroCopula": NeuroCopula(transforms=4, hidden_features=(64, 64), seed=0).fit(
        train, epochs=600, patience=60, split_method="random", verbose=False),
    "Vine (parametric)": VineCopula(family_set="parametric").fit(train),
    "Vine (Gaussian only)": VineCopula(family_set="gaussian").fit(train),
}

for name, m in models.items():
    print(f"{name:22s} held-out copula loglik = {m.score(test, copula=True):+.4f}")

Both expose `fit`, `sample`, `sample_uniform`, `log_prob`, `copula_log_prob`, `score`,
`tail_dependence`, `tail_dependence_curve` and `joint_exceedance_probability`. Every number
below is therefore computed the same way for both models.

**Why `copula_log_prob` and not `log_prob`?** Both models use the *same* empirical marginal
transform, so the marginal half of the factorisation is identical by construction. Scoring the
copula alone isolates the only thing that differs, and it is exact for both — no density
estimate involved.

## The vine's advantage: interpretability

A vine tells you *what* it found. A flow cannot.

In [ ]:
vine = models["Vine (parametric)"]
print("pair copula families selected, by tree:")
for level, families in enumerate(vine.pair_families()):
    print(f"  tree {level}: {families}")
print(f"\nKendall's tau per pair copula: {vine.taus()}")
print(f"total parameters: {vine.fit_report_['n_parameters']:.0f}")
print()
print("It recovered the Clayton family from Clayton data, with tau close to the true 0.5.")
print("That is a genuine, checkable statement about the dependence -- a flow offers nothing")
print("comparable; you can only probe it by sampling.")

## Case 1 — a family the vine has

Clayton data. The vine's catalogue contains Clayton, so this is the vine's home turf and we
should expect it to win.

In [ ]:
truth = datasets.theoretical_tail_dependence("clayton", theta=2.0)
rows = []
for name, m in models.items():
    u = m.sample_uniform(200_000)
    rows.append({
        "model": name,
        "held-out loglik": m.score(test, copula=True),
        "lambda_L": metrics.tail_dependence(u.x1, u.x2, q=0.02, tail="lower"),
        "lambda_U": metrics.tail_dependence(u.x1, u.x2, q=0.02, tail="upper"),
    })
res = pd.DataFrame(rows).set_index("model")
res["lambda_L error"] = (res["lambda_L"] - truth["lower"]).abs()
res["lambda_U error"] = (res["lambda_U"] - truth["upper"]).abs()
print(f"truth: lambda_L = {truth['lower']:.3f}, lambda_U = {truth['upper']:.3f}\n")
res.round(3)

The parametric vine edges out the flow on log-likelihood, as expected — it is fitting one
parameter where the flow fits ~29,000, so when the family is right it is simply more
efficient. **The flow gets very close without being told the family**, which is the claim
being made for it; it is not the claim that it wins.

The Gaussian-only vine is the cautionary row: it reports λ_L ≈ λ_U on data whose true values
are 0.707 and 0.0. That failure is **structural** — an elliptical pair copula cannot represent
tail asymmetry, and no amount of data fixes it.

In [ ]:
fig = ncplot.plot_model_comparison(
    train, {"NeuroCopula": models["NeuroCopula"], "Vine": models["Vine (Gaussian only)"]},
    "x1", "x2", n=4000, theoretical=truth["lower"],
)
plt.show()

Look at the scatter panels. The data and the flow both crowd the bottom-left corner; the
Gaussian vine spreads symmetrically. The curve panel makes the consequence explicit: the flow
tracks the theoretical line, the Gaussian vine sits far below it and decays.

## Case 2 — a structure no family fits

Now the two-regime mixture: mostly calm and weakly correlated, with a crisis minority that is
strongly correlated and larger in scale. This is not in any standard catalogue, and it is a
reasonable caricature of how real markets behave.

In [ ]:
mix = datasets.make_mixed_regime(n=4000, rho_calm=0.2, rho_crisis=0.9,
                                 crisis_frac=0.15, seed=0)
mix_train, mix_test = _split(mix, 0.25, seed=0)

mix_models = {
    "NeuroCopula": NeuroCopula(transforms=4, hidden_features=(64, 64), seed=0).fit(
        mix_train, epochs=600, patience=60, split_method="random", verbose=False),
    "Vine (parametric)": VineCopula(family_set="parametric").fit(mix_train),
}

u_data = mix_models["NeuroCopula"].pseudo_observations(mix_test)
rows = []
for name, m in mix_models.items():
    u = m.sample_uniform(200_000)
    rows.append({
        "model": name,
        "held-out loglik": m.score(mix_test, copula=True),
        "lambda_L": metrics.tail_dependence(u.x1, u.x2, q=0.02, tail="lower"),
        "lambda_U": metrics.tail_dependence(u.x1, u.x2, q=0.02, tail="upper"),
        "energy distance": metrics.energy_distance(u, u_data),
    })
rows.append({
    "model": "-- data --",
    "held-out loglik": np.nan,
    "lambda_L": metrics.tail_dependence(u_data.x1, u_data.x2, q=0.02, tail="lower"),
    "lambda_U": metrics.tail_dependence(u_data.x1, u_data.x2, q=0.02, tail="upper"),
    "energy distance": 0.0,
})
pd.DataFrame(rows).set_index("model").round(4)

Compare each model's λ_L against the `-- data --` row. The vine, forced to pick one parametric
family for the pair, lands on a compromise that fits the calm majority and badly understates
the joint-tail behaviour created by the crisis component. The flow, free of family
constraints, gets substantially closer — and wins on held-out likelihood as well.

**This is the case for a neural copula.** Not that it beats a vine everywhere, but that when
the dependence has no parametric name, it does not have to be forced into one.

In [ ]:
frames = {
    "data": u_data,
    "NeuroCopula": mix_models["NeuroCopula"].sample_uniform(60_000),
    "Vine": mix_models["Vine (parametric)"].sample_uniform(60_000),
}
fig = ncplot.plot_copula_scatter(frames, "x1", "x2", n=4000)
plt.show()
fig = ncplot.plot_tail_concentration(frames, "x1", "x2")
plt.show()

The data's copula-scale scatter shows a dense diagonal core with heavy corner clustering — the
mixture's two regimes superimposed. The flow reproduces both features. The vine produces a
single smooth shape with visibly thinner corners.

Both panels of the tail concentration figure tell the same story on each side.

## Cost

Accuracy is not the only axis.

In [ ]:
rows = []
for name, ctor in [
    ("NeuroCopula", lambda: NeuroCopula(transforms=4, hidden_features=(64, 64), seed=0).fit(
        mix_train, epochs=600, patience=60, split_method="random", verbose=False)),
    ("Vine (parametric)", lambda: VineCopula(family_set="parametric").fit(mix_train)),
]:
    t0 = time.perf_counter(); m = ctor(); fit_s = time.perf_counter() - t0
    t0 = time.perf_counter(); m.sample_uniform(100_000); sample_s = time.perf_counter() - t0
    n_par = (sum(p.numel() for p in m.flow.parameters()) if isinstance(m, NeuroCopula)
             else m.vine.npars)
    rows.append({"model": name, "fit seconds": fit_s,
                 "100k draws (s)": sample_s, "parameters": n_par})
pd.DataFrame(rows).set_index("model").round(3)

The vine fits in a fraction of a second; the flow takes tens of seconds and carries four
orders of magnitude more parameters. Whether that matters depends entirely on your setting: it
is negligible if you fit once and query all day, and prohibitive if you refit per instrument
on a schedule.

## Fit both — that is the point

Since they share an API, the sensible workflow is not to choose in advance but to fit both and
compare on held-out data.

In [ ]:
def compare(data, label, epochs=600):
    tr, te = _split(data, 0.25, seed=0)
    flow = NeuroCopula(transforms=4, hidden_features=(64, 64), copula_mode=True,
                       marginal_density="none", sampling_mode="coupling", seed=0).fit(
        tr, epochs=epochs, patience=50, split_method="random", verbose=False)
    vine = VineCopula(family_set="parametric", marginal_density="none").fit(tr)
    f, v = flow.score(te, copula=True), vine.score(te, copula=True)
    return {"dataset": label, "flow": f, "vine": v, "difference": f - v,
            "winner": "flow" if f > v else "vine"}

summary = pd.DataFrame([
    compare(datasets.make_gaussian_copula(n=4000, rho=0.7, seed=1), "Gaussian copula"),
    compare(datasets.make_clayton(n=4000, theta=2.0, seed=1), "Clayton"),
    compare(datasets.make_t_copula(n=4000, rho=0.7, dof=4, seed=1), "Student-t"),
    compare(datasets.make_mixed_regime(n=4000, seed=1), "Two-regime mixture"),
    compare(datasets.make_mixed_regime(n=4000, d=5, seed=1), "Mixture, 5-D"),
    compare(datasets.make_asset_panel(n=4000, seed=1), "Asset panel, 5-D"),
]).set_index("dataset")
summary.round(4)

The pattern is consistent with the full benchmark in this repository: the vine wins wherever
its catalogue contains the generating family, the flow wins wherever it does not, and the
margins are small in the first case and larger in the second.

## When to use which

| Use a vine copula when | Use `neurocopula` when |
|---|---|
| A standard family plausibly fits | Dependence is asymmetric, multi-modal, or regime-switching |
| Data is limited relative to $d$ | You have enough rows for a flexible model |
| You need interpretable pair-by-pair structure | You care most about joint-tail accuracy |
| Fitting must be fast | Fit cost is amortised over many queries |
| You must justify the model to a regulator | Predictive accuracy is the criterion |

## Takeaways

1. **A vine with the right family is hard to beat**, and is more sample-efficient.
2. **A vine restricted to the wrong family fails structurally** — a Gaussian pair copula
   cannot produce tail asymmetry at any sample size.
3. **The flow's advantage appears when no family fits**, which is when it is worth its cost.
4. **The flow gets close on the vine's home turf without being told the family.**
5. **Fit both.** They share an API, and held-out `score(..., copula=True)` settles it.

Continue to [`../benchmarks/benchmark_suite.ipynb`](../benchmarks/benchmark_suite.ipynb) for
the full multi-seed benchmark behind these claims.